# Load libraries

In [ ]:
# %pip install mlflow

In [ ]:
# %pip install python-dateutil

In [ ]:
import sys
# adding to the path variables the one folder higher (locally, not changing system variables)
sys.path.append("..")
import pandas as pd
import numpy as np
import warnings
import mlflow
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta
from dateutil.relativedelta import *
import itertools
from itertools import combinations
from collections import defaultdict
import pprint
from ydata_profiling import ProfileReport

from prophet import Prophet
from prophet.plot import plot
from prophet.plot import plot_yearly
from prophet.make_holidays import make_holidays_df
from prophet.plot import plot_forecast_component
from prophet.plot import add_changepoints_to_plot
from prophet.utilities import regressor_coefficients
from prophet.diagnostics import cross_validation
from prophet.diagnostics import performance_metrics
from prophet.plot import plot_cross_validation_metric

from sklearn.metrics import r2_score
from sklearn.metrics import mean_squared_error


warnings.filterwarnings('ignore')

# Display all columns
pd.set_option('display.max_columns', 10000)
pd.set_option('display.max_colwidth', 10000)
pd.options.display.max_rows = 100000
pd.options.display.width = 100000

from IPython.display import display


In [ ]:
def is_weekend(ds):
    date = pd.to_datetime(ds)
    return (date.dayofweek == 5 or date.dayofweek == 6)

In [ ]:
def info_describe(df, N_MOST_FREQ=5):
    
    # Display top-N_MOST_FREQ most frequent unique values

    # Get basic statistics for numeric columns using df.describe()
    describe_numeric = df.describe().transpose()

    # Combine the information into a single DataFrame
    feature_info_list = []

    for i, col in enumerate(df.columns):
        
        data_type = df[col].dtype
        non_null_count = df[col].count()
        unique_count = df[col].nunique()
        top = df[col].mode().iloc[0] if non_null_count > 0 else None
        frequency = df[col].value_counts().to_dict() if non_null_count > 0 else None
        if data_type != 'object' and data_type != 'datetime64[ns]' and data_type != 'datetime64[ns, pytz.FixedOffset(60)]':  # Numeric columns
            describe_numeric_values = describe_numeric.loc[col].tolist()
        else:
            describe_numeric_values = [None] * 8  # Placeholder values for non-numeric columns (number of columns from output of df.describe().transpose())

        if frequency is not None and len(frequency) > N_MOST_FREQ:  # Limit to top-N_MOST_FREQ most frequent unique values
            top_n_freq = dict(list(frequency.items())[:N_MOST_FREQ])
            frequency_str = str(top_n_freq)[:-1] + ', ...}'
        else:
            frequency_str = str(frequency)

        feature_info = {
            # 'Feature Index': i,
            'Feature Name': col,
            'Data Type': data_type,
            'Non-Null Count': non_null_count,
            'Num of Unique Values': unique_count,
            'Mean': describe_numeric_values[1],
            'Std': describe_numeric_values[2],
            'Min': describe_numeric_values[3],
            '25%': describe_numeric_values[4],
            '50%': describe_numeric_values[5],
            '75%': describe_numeric_values[6],
            'Max': describe_numeric_values[7],
            'Most Frequent Value': top,
            'Most Frequent Unique Values': frequency_str
        }
        feature_info_list.append(feature_info)

    # Create the final DataFrame
    feature_info_df = pd.DataFrame(feature_info_list)

    # Print the final DataFrame using display()
    print("\nDataFrame Description and Unique Value Counts:")

    # No wrapping for the display of the final DataFrame
    feature_info_df_nowrap = feature_info_df.style.set_table_styles([dict(selector="td", props=[('white-space', 'nowrap')])])
    display(feature_info_df_nowrap)

In [ ]:
# Customize a palette

NF_ORANGE = '#ff5a36'
NF_BLUE = '#163251'
cmaps_hex = ['#193251','#FF5A36','#696969', '#7589A2','#FF5A36', '#DB6668']
sns.set_palette(palette=cmaps_hex)
sns_c = sns.color_palette(palette=cmaps_hex)

In [ ]:
sns_c

# Import 'df_daily.csv' (Germany's daily energy generation)

In [ ]:
df_orig = pd.read_csv('data/df_daily.csv')

df_orig.head(2)
df_orig.tail(2)

In [ ]:
df_orig.info()

In [ ]:
df_orig['dt_date'] = pd.to_datetime(df_orig['dt_date'])
#df.columns = ['ds', 'y']

In [ ]:
df_orig.info()
df_orig.head(2)

In [ ]:
df = df_orig.copy()

In [ ]:
df.rename(columns={
    "dt_date": "ds",
    "solar_gwh": "y"
}, 
inplace=True)

In [ ]:
df_d = df.copy()
df_d.info()

# Prophet on 'df_m_all'

## Import 'df_m_all'

In [ ]:
df_orig_m = pd.read_csv('data/df_m_all.csv')

df_orig_m.head(2)
df_orig_m.tail(2)

In [ ]:
# Convert 'dt_date' column to datetime objects pointing to the last day of the month

# df_orig_m['dt_date'] = pd.to_datetime(df_orig_m['dt_date'], format='%Y-%m')
df_orig_m['dt_date'] = pd.to_datetime(df_orig_m['dt_date']).dt.to_period('M').dt.to_timestamp('M').dt.to_period('M').dt.end_time

In [ ]:
df_orig_m.head(2)

In [ ]:
df_m = df_orig_m.copy()

In [ ]:
df_m.info()

In [ ]:
df_m.rename(columns={
    "dt_date": "ds",
    "solar_twh": "y"
}, 
inplace=True)

In [ ]:
df_m.info()

## Very first attempt

In [ ]:
model = Prophet()
model.fit(df_m)

In [ ]:
future = model.make_future_dataframe(periods=12 * 1)
forecast = model.predict(future)

In [ ]:
fig = model.plot(forecast, xlabel='Date',
                 ylabel='Daily solar [TWh]')
plt.title("Germany's monthly solar energy generation")
plt.show()

In [ ]:
forecast.head(3).T

## Forecast for n months

### Without additional input features

In [ ]:
model = Prophet(seasonality_mode='multiplicative',
                yearly_seasonality=4,
                changepoint_prior_scale=0.5)

In [ ]:
# Reserve final n months of training data for forecasting

n_months_forecast = 12  # Use 30 months' data for forecasting

# Ensure the end date is included in the calculation by explicitly setting the 'day' parameter to 31 in the 'relativedelta' function
print(df_m['ds'].max(), df_m['ds'].max() - relativedelta(months=n_months_forecast, day=31))

date_last = df_m['ds'].max()
date_cutoff = date_last - relativedelta(months=n_months_forecast, day=31)

train = df_m[df_m['ds'] <= date_cutoff]
# Alternatively
# train = df_m[df_m['ds'].dt.year < 2021]
# test_df = df_m[df_m['ds'].dt.year >= 2021]

test_df = df_m[df_m['ds'] > date_cutoff]

In [ ]:
# Calculate number of days for forecasting

# start_date_temp = '2023-04-30'
end_date_temp = '2023-06-29'
n_days_temp = 61

# Now calculate the cutoff date
end_date_temp = pd.to_datetime(end_date_temp)

date_last_temp = end_date_temp
       
date_cutoff_temp = date_last_temp - relativedelta(days=n_days_temp)

print(f"{date_cutoff_temp} will be the cutoff date.")

In [ ]:
model.fit(train)

In [ ]:
# Forecast future and plot forecast using Prophet's built-in plot function

future = model.make_future_dataframe(periods=12*1, freq='M')  # Prophet will automatically determine the end date of each month
# future = model.make_future_dataframe(periods=days_forecast, freq='D')
forecast = model.predict(future)

fig = model.plot(forecast)

add_changepoints_to_plot(fig.gca(), model, forecast, cp_linestyle='')

plt.scatter(test_df['ds'], test_df['y'], color='brown', label='Test dataset', alpha=0.5)

plt.show()

In [ ]:
forecast_train = forecast[forecast['ds'] <= date_cutoff]
forecast_test = forecast[forecast['ds'] > date_cutoff]

print(f"Shape of 'forecast_train' DataFrame: {forecast_train.shape}.")
print(f"Shape of 'forecast_test' DataFrame: {forecast_test.shape}.")

In [ ]:
# Plot the training and test parts of 'forecast' using DataFrame's 'plot' method

# Plots for actual values and predictions of training dataset
# fig = model.plot(forecast_train)  # Alternatively
fig = plot(model, forecast_train)

plt.gca().lines[1].set_linestyle('-')  # Change linestyle of the forecast line

ax = fig.gca()

# forecast_train.plot(x='ds', y='yhat', ax=ax, label='Predictions for train dataset', color=sns_c[0], linestyle='-')
# ax.fill_between(forecast_train['ds'], forecast_train['yhat_lower'], forecast_train['yhat_upper'], color=sns_c[0], alpha=0.2)
# plt.scatter(train['ds'], train['y'], color='black', label='Training dataset', alpha=0.5)
forecast_train.plot(x='ds', y='trend', ax=ax, label='Forecasts for training dataset: trend', color='blue', linestyle='--')

# Plots for actual values and predictions of test dataset
plt.scatter(test_df['ds'], test_df['y'], color='brown', label='Test dataset', alpha=0.5, s=10)
forecast_test.plot(x='ds', y='yhat', ax=ax, label='Forecasts for test dataset', color='orange', linestyle='-')
ax.fill_between(forecast_test['ds'], forecast_test['yhat_lower'], forecast_test['yhat_upper'], color='orange', alpha=0.2)
forecast_test.plot(x='ds', y='trend', ax=ax, label='Forecasts for test dataset: trend', color='red', linestyle='--')

# add_changepoints_to_plot(ax, model, forecast_train, cp_linestyle='', cp_color='blue')
# add_changepoints_to_plot(ax, model, forecast_test, cp_linestyle='', cp_color='red')

plt.legend()
plt.show()

In [ ]:
# Plot the training and test parts of 'forecast' using Matplotlib

# Plots for actual values and predictions of training dataset
fig, ax = plt.subplots(figsize=(10, 6))

# Plots for actual values and predictions of training dataset
ax.scatter(train['ds'], train['y'], label='Training dataset', color='black', alpha=0.5, s=10)
ax.plot(forecast_train['ds'], forecast_train['yhat'], label='Predictions for training dataset', color='navy', linestyle='-')
ax.fill_between(forecast_train['ds'], forecast_train['yhat_lower'], forecast_train['yhat_upper'], color='navy', alpha=0.2)
ax.plot(forecast_train['ds'], forecast_train['trend'], label='Predictions for training dataset: trend', color='blue', linestyle='--')

# Plots for actual values and predictions of test dataset
ax.scatter(test_df['ds'], test_df['y'], label='Test dataset', color='brown', alpha=0.5, s=10)
ax.plot(forecast_test['ds'], forecast_test['yhat'], label='Predictions for test dataset', color='orange', linestyle='-')
ax.fill_between(forecast_test['ds'], forecast_test['yhat_lower'], forecast_test['yhat_upper'], color='orange', alpha=0.2)
ax.plot(forecast_test['ds'], forecast_test['trend'], label='Predictions for test dataset: trend', color='red', linestyle='--')

# Customize legend
ax.legend(loc='upper left')

# Set labels and title
ax.set_xlabel('Date')
ax.set_ylabel("Solar energy generation [TWh]")
ax.set_title("Germany's monthly solar energy generation: actual vs. predicted")

plt.legend()
plt.show()


In [ ]:
print(future.tail(132))
# print(forecast)
print(forecast.shape)

### With additional features

In [ ]:
df_m.info()

In [ ]:
# Model with additional features/regressors
# Consider automating the following procedure by building a function

model_af = Prophet(seasonality_mode='multiplicative',
                yearly_seasonality=4,
                changepoint_prior_scale=0.5)

model_af.add_regressor('solar_gw')
model_af.add_regressor('mean_air_temperature_de_c')
model_af.add_regressor('mean_sunshine_duration_de_minute')
model_af.add_regressor('mean_precipitation_de_mm')

In [ ]:
# Reserve final n months of training data for forecasting

n_months_forecast = 12  # Use 30 months' data for forecasting

# Ensure the end date is included in the calculation by explicitly setting the 'day' parameter to 31 in the 'relativedelta' function
date_last = df_m['ds'].max()
date_cutoff = date_last - relativedelta(months=n_months_forecast, day=31)

train = df_m[df_m['ds'] <= date_cutoff]
# Alternatively
# train = df_m[df_m['ds'].dt.year < 2021]
# test_df = df_m[df_m['ds'].dt.year >= 2021]

test_df = df_m[df_m['ds'] > date_cutoff]

In [ ]:
model_af.fit(train)

In [ ]:
# Forecast future and plot forecast using Prophet's built-in plot function

future_af = model_af.make_future_dataframe(periods=12*1, freq='M')  # Prophet will automatically determine the end date of each month

future_af['solar_gw'] = df_m['solar_gw']
future_af['mean_air_temperature_de_c'] = df_m['mean_air_temperature_de_c']
future_af['mean_sunshine_duration_de_minute'] = df_m['mean_sunshine_duration_de_minute']
future_af['mean_precipitation_de_mm'] = df_m['mean_precipitation_de_mm']

forecast_af = model_af.predict(future_af)

fig = model_af.plot(forecast_af)

add_changepoints_to_plot(fig.gca(), model_af, forecast_af, cp_linestyle='')

plt.scatter(test_df['ds'], test_df['y'], color='brown', label='Test dataset', alpha=0.5)

plt.show()

### Define a function 'prophet_af' for Prophet-based modeling

In [ ]:
# Function for Prophet-based modeling

# NOTE: before applying this function, make sure:
# 1. data preprocessing is done
# 2. column names 'ds' and 'y' are properly assigned to the date series and the target feature, respectively
# 3. 'ds' already has a datatype of 'datetime64[ns]'

def prophet_af(df, seasonality_mode='multiplicative', yearly_seasonality=4, changepoint_prior_scale=0.5, list_added_regressor=[], freq='M', n_timeunits_forecast=12):

    model_af = Prophet(seasonality_mode=seasonality_mode,
                yearly_seasonality=yearly_seasonality,
                changepoint_prior_scale=changepoint_prior_scale)

    if list_added_regressor:
        for item_regressor in list_added_regressor:
            model_af.add_regressor(item_regressor)

    # If 'freq' equals 'M', 'ds' is essentially 'dt_date' with a format of 'yyyy-mm' and a datatype of 'datetime64[ns]'
    # If 'freq' equals 'D', 'ds' is essentially 'dt_date' with a format of 'yyyy-mm-dd' and a datatype of 'datetime64[ns]'
    if freq == 'M':
        # Convert 'dt_date' column to datetime objects pointing to the last day of the month
        df['ds'] = pd.to_datetime(df['ds']).dt.to_period('M').dt.to_timestamp('M').dt.to_period('M').dt.end_time
        date_last = df['ds'].max()
        # Ensure the end date is included in the calculation by explicitly setting the 'day' parameter to 31 in the 'relativedelta' function
        date_cutoff = date_last - relativedelta(months=n_timeunits_forecast, day=31)
    elif freq == 'D':
        date_last = df['ds'].max()
        date_cutoff = date_last - relativedelta(days=n_timeunits_forecast)
    else:
        raise ValueError("Value for argument 'freq' can only be 'M' or 'D'!")

    train = df[df['ds'] <= date_cutoff]
    test_df = df[df['ds'] > date_cutoff]

    model_af.fit(train)

    future_af = model_af.make_future_dataframe(periods=n_timeunits_forecast, freq=freq)  # Prophet will automatically determine the end date of each month

    if list_added_regressor:
        for item_regressor in list_added_regressor:
            future_af[item_regressor] = df[item_regressor]

    forecast_af = model_af.predict(future_af)

    forecast_train = forecast_af[forecast_af['ds'] <= date_cutoff]
    forecast_test = forecast_af[forecast_af['ds'] > date_cutoff]

    print("====="*12)
    print(f"          {'DataFrame Name':<30}{'Shape':<12}")
    print("         ","------"*3,"          ","------"*2)
    print(f"          {'forecast_af':<30}{str(forecast_af.shape):<12}")
    print(f"          {'future_af':<30}{str(future_af.shape):<12}")
    print(f"          {'train':<30}{str(train.shape):<12}")
    print(f"          {'test_df':<30}{str(test_df.shape):<12}")
    print(f"          {'forecast_train':<30}{str(forecast_train.shape):<12}")
    print(f"          {'forecast_test':<30}{str(forecast_test.shape):<12}")


    return forecast_af, future_af, train, test_df, forecast_train, forecast_test



### Run the 'prophet_af' function

In [ ]:
# Run the 'prophet_af' function

forecast_af, future_af, train, test_df, forecast_train, forecast_test = prophet_af(df_m, seasonality_mode='multiplicative', yearly_seasonality=4, changepoint_prior_scale=0.5, list_added_regressor=['solar_gw', 'mean_air_temperature_de_c', 'mean_sunshine_duration_de_minute', 'mean_precipitation_de_mm'], freq='M', n_timeunits_forecast=12)

In [ ]:
# Plot the training and test parts of 'forecast' using Matplotlib

fig, ax = plt.subplots(figsize=(10, 6))

# Plots for actual values and predictions of training dataset
ax.scatter(train['ds'], train['y'], label='Training dataset', color='black', alpha=0.5, s=10)
ax.plot(forecast_train['ds'], forecast_train['yhat'], label='Predictions for training dataset', color='navy', linestyle='-')
ax.fill_between(forecast_train['ds'], forecast_train['yhat_lower'], forecast_train['yhat_upper'], color='navy', alpha=0.2)
ax.plot(forecast_train['ds'], forecast_train['trend'], label='Predictions for training dataset: trend', color='blue', linestyle='--')

# Plots for actual values and predictions of test dataset
ax.scatter(test_df['ds'], test_df['y'], label='Test dataset', color='brown', alpha=0.5, s=10)
ax.plot(forecast_test['ds'], forecast_test['yhat'], label='Predictions for test dataset', color='orange', linestyle='-')
ax.fill_between(forecast_test['ds'], forecast_test['yhat_lower'], forecast_test['yhat_upper'], color='orange', alpha=0.2)
ax.plot(forecast_test['ds'], forecast_test['trend'], label='Predictions for test dataset: trend', color='red', linestyle='--')

# Customize legend
ax.legend(loc='upper left')

# Set labels and title
ax.set_xlabel('Date')
ax.set_ylabel("Solar energy generation [TWh]")
ax.set_title("Germany's monthly solar energy generation: actual vs. predicted")

plt.legend()
plt.show()

In [ ]:
# Zoom-in plot

fig, ax = plt.subplots(figsize=(10, 6))

# Define the number of steps for zoom-in
n_steps = 5

# Determine the range of dates for zoom-in plot of train DataFrame
train_zoom_start = train['ds'].max() - pd.DateOffset(months=n_steps)
train_zoom_end = train['ds'].max()

# Determine the range of dates for zoom-in plot of test_df DataFrame
test_zoom_start = test_df['ds'].min()
test_zoom_end = test_df['ds'].min() + pd.DateOffset(months=n_steps)

# Filter data for zoom-in plot
train_zoom = train[(train['ds'] >= train_zoom_start) & (train['ds'] <= train_zoom_end)]
test_zoom = test_df[(test_df['ds'] >= test_zoom_start) & (test_df['ds'] <= test_zoom_end)]

# Determine the range of dates for zoom-in plot of forecast_train DataFrame
forecast_train_zoom_start = forecast_train['ds'].max() - pd.DateOffset(months=n_steps)
forecast_train_zoom_end = forecast_train['ds'].max()

# Determine the range of dates for zoom-in plot of test_df DataFrame
forecast_test_zoom_start = forecast_test['ds'].min()
forecast_test_zoom_end = forecast_test['ds'].min() + pd.DateOffset(months=n_steps)

# Filter data for zoom-in plot
forecast_train_zoom = forecast_train[(forecast_train['ds'] >= forecast_train_zoom_start) & (forecast_train['ds'] <= forecast_train_zoom_end)]
forecast_test_zoom = forecast_test[(forecast_test['ds'] >= forecast_test_zoom_start) & (forecast_test['ds'] <= forecast_test_zoom_end)]


# Plots for actual values and predictions of training dataset
ax.scatter(train_zoom['ds'], train_zoom['y'], label='Training dataset', color='black', alpha=0.5, s=10)
ax.plot(forecast_train_zoom['ds'], forecast_train_zoom['yhat'], label='Predictions for training dataset', color='navy', linestyle='-')
ax.fill_between(forecast_train_zoom['ds'], forecast_train_zoom['yhat_lower'], forecast_train_zoom['yhat_upper'], color='navy', alpha=0.2)
ax.plot(forecast_train_zoom['ds'], forecast_train_zoom['trend'], label='Predictions for training dataset: trend', color='blue', linestyle='--')

# Plots for actual values and predictions of test dataset
ax.scatter(test_zoom['ds'], test_zoom['y'], label='Test dataset', color='brown', alpha=0.5, s=10)
ax.plot(forecast_test_zoom['ds'], forecast_test_zoom['yhat'], label='Predictions for test dataset', color='orange', linestyle='-')
ax.fill_between(forecast_test_zoom['ds'], forecast_test_zoom['yhat_lower'], forecast_test_zoom['yhat_upper'], color='orange', alpha=0.2)
ax.plot(forecast_test_zoom['ds'], forecast_test_zoom['trend'], label='Predictions for test dataset: trend', color='red', linestyle='--')

# Customize legend
ax.legend(loc='upper left')

# Set labels and title
ax.set_xlabel('Date')
ax.set_ylabel("Solar energy generation [TWh]")
ax.set_title("Germany's monthly solar energy generation: actual vs. predicted")

plt.legend()
plt.show()



### Function for the plotting of training and test parts of 'forecast'

In [ ]:
# Function for the plotting of the training and test parts of 'forecast' using Matplotlib

# Plots for actual values and predictions of training dataset
def plot_prophet_forecast(train, test_df, forecast_train, forecast_test, xlabel, ylabel, title):

    fig, ax = plt.subplots(figsize=(10, 6))

    # Plots for actual values and predictions of training dataset
    ax.scatter(train['ds'], train['y'], label='Training dataset', color='black', alpha=0.5, s=10)
    ax.plot(forecast_train['ds'], forecast_train['yhat'], label='Predictions for training dataset', color='navy', linestyle='-')
    ax.fill_between(forecast_train['ds'], forecast_train['yhat_lower'], forecast_train['yhat_upper'], color='navy', alpha=0.2)
    ax.plot(forecast_train['ds'], forecast_train['trend'], label='Predictions for training dataset: trend', color='blue', linestyle='--')

    # Plots for actual values and predictions of test dataset
    ax.scatter(test_df['ds'], test_df['y'], label='Test dataset', color='brown', alpha=0.5, s=10)
    ax.plot(forecast_test['ds'], forecast_test['yhat'], label='Predictions for test dataset', color='orange', linestyle='-')
    ax.fill_between(forecast_test['ds'], forecast_test['yhat_lower'], forecast_test['yhat_upper'], color='orange', alpha=0.2)
    ax.plot(forecast_test['ds'], forecast_test['trend'], label='Predictions for test dataset: trend', color='red', linestyle='--')

    # Customize legend
    ax.legend(loc='upper left')

    # Set labels and title
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    ax.set_title(title)

    plt.legend()
    plt.show()

In [ ]:
# Run the 'plot_prophet_forecast' function

plot_prophet_forecast(train, test_df, forecast_train, forecast_test, "Date", "Solar energy generation [TWh]", "Germany's monthly solar energy generation: actual vs. predicted")

### Function for the zoom-in plot of training and test parts of 'forecast'

In [ ]:
### Function for drawing the zoom-in plot of training and test parts of 'forecast'

def plot_prophet_forecast_zoomin(train, test_df, forecast_train, forecast_test, xlabel, ylabel, title, n_steps=6, time_unit='M'):
    
    fig, ax = plt.subplots(figsize=(10, 6))

    # Determine the range of dates for zoom-in plot of train DataFrame
    # For the time being, time_uint can only be 'M' (month) or 'D' (day)
    if time_unit == 'M':
        train_zoom_start = train['ds'].max() - pd.DateOffset(months=n_steps)
        test_zoom_end = test_df['ds'].min() + pd.DateOffset(months=n_steps)
        forecast_train_zoom_start = forecast_train['ds'].max() - pd.DateOffset(months=n_steps)
        forecast_test_zoom_end = forecast_test['ds'].min() + pd.DateOffset(months=n_steps)
    elif time_unit == 'D':
        train_zoom_start = train['ds'].max() - pd.DateOffset(days=n_steps)
        test_zoom_end = test_df['ds'].min() + pd.DateOffset(days=n_steps)
        forecast_train_zoom_start = forecast_train['ds'].max() - pd.DateOffset(days=n_steps)
        forecast_test_zoom_end = forecast_test['ds'].min() + pd.DateOffset(days=n_steps)
    else:
        raise ValueError("For the time being, value for argument 'time_unit' can only be 'M' or 'D'!")

    train_zoom_end = train['ds'].max()

    # Determine the range of dates for zoom-in plot of test_df DataFrame
    test_zoom_start = test_df['ds'].min()

    # Filter data for zoom-in plot
    train_zoom = train[(train['ds'] >= train_zoom_start) & (train['ds'] <= train_zoom_end)]
    test_zoom = test_df[(test_df['ds'] >= test_zoom_start) & (test_df['ds'] <= test_zoom_end)]

    # Determine the range of dates for zoom-in plot of forecast_train DataFrame
    
    forecast_train_zoom_end = forecast_train['ds'].max()

    # Determine the range of dates for zoom-in plot of test_df DataFrame
    forecast_test_zoom_start = forecast_test['ds'].min()
    
    # Filter data for zoom-in plot
    forecast_train_zoom = forecast_train[(forecast_train['ds'] >= forecast_train_zoom_start) & (forecast_train['ds'] <= forecast_train_zoom_end)]
    forecast_test_zoom = forecast_test[(forecast_test['ds'] >= forecast_test_zoom_start) & (forecast_test['ds'] <= forecast_test_zoom_end)]

    # Plots for actual values and predictions of training dataset
    # ax.scatter(train_zoom['ds'], train_zoom['y'], label='Training dataset', color='black', alpha=0.5, s=10)
    ax.plot(train_zoom['ds'], train_zoom['y'], label='Training dataset', color='black', linestyle='-', linewidth=1)
    ax.plot(forecast_train_zoom['ds'], forecast_train_zoom['yhat'], label='Predictions for training dataset', color='navy', linestyle='--', linewidth=1)
    ax.fill_between(forecast_train_zoom['ds'], forecast_train_zoom['yhat_lower'], forecast_train_zoom['yhat_upper'], color='navy', alpha=0.2)
    ax.plot(forecast_train_zoom['ds'], forecast_train_zoom['trend'], label='Predictions for training dataset: trend', color='blue', linestyle='--')

    # Plots for actual values and predictions of test dataset
    # ax.scatter(test_zoom['ds'], test_zoom['y'], label='Test dataset', color='brown', alpha=0.5, s=10)
    ax.plot(test_zoom['ds'], test_zoom['y'], label='Test dataset', color='brown', linestyle='-', linewidth=1)
    ax.plot(forecast_test_zoom['ds'], forecast_test_zoom['yhat'], label='Predictions for test dataset', color='firebrick', linestyle='--', linewidth=1)
    ax.fill_between(forecast_test_zoom['ds'], forecast_test_zoom['yhat_lower'], forecast_test_zoom['yhat_upper'], color='orange', alpha=0.2)
    ax.plot(forecast_test_zoom['ds'], forecast_test_zoom['trend'], label='Predictions for test dataset: trend', color='red', linestyle='--')

    # Customize legend
    ax.legend(loc='upper left')

    # Set labels and title
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    ax.set_title(title)

    plt.legend()
    plt.show()

In [ ]:
# Run the 'plot_prophet_forecast_zoomin' function

plot_prophet_forecast_zoomin(train, test_df, forecast_train, forecast_test, "Date", "Solar energy generation [TWh]", "Germany's monthly solar energy generation: actual vs. predicted", 6, 'M')

### Performance metrics

In [ ]:
# Calculate R-squared and RMSE for training data
r2_train = r2_score(train['y'], forecast_train['yhat'][:len(train)])
rmse_train = np.sqrt(mean_squared_error(train['y'], forecast_train['yhat'][:len(train)]))
# Calculate R-squared and RMSE for test data
r2_test = r2_score(test_df['y'], forecast_test['yhat'][:len(test_df)])
rmse_test = np.sqrt(mean_squared_error(test_df['y'], forecast_test['yhat'][:len(test_df)]))

print("====="*12)
print(f"          {'R2 score':<30}{'RMSE':<12}")
print("         ","------"*3,"          ","------"*2)
print(f"Train:    {r2_train:<30.4f}{rmse_train:<12.4f}")
print(f"Test:     {r2_test:<30.4f}{rmse_test:<12.4f}")


### Function for the calculation of R-squared and RMSE

In [ ]:
# Function for the calculation of R-squared and RMSE

def prophet_metric_train_test(train, test_df, forecast_train, forecast_test, verbose=0):

    # Calculate R-squared and RMSE for training data
    r2_train = r2_score(train['y'], forecast_train['yhat'][:len(train)])
    rmse_train = np.sqrt(mean_squared_error(train['y'], forecast_train['yhat'][:len(train)]))
    # Calculate R-squared and RMSE for test data
    r2_test = r2_score(test_df['y'], forecast_test['yhat'][:len(test_df)])
    rmse_test = np.sqrt(mean_squared_error(test_df['y'], forecast_test['yhat'][:len(test_df)]))

    if verbose != 0:
        print("====="*12)
        print(f"          {'R2 score':<30}{'RMSE':<12}")
        print("         ","------"*3,"          ","------"*2)
        print(f"Train:    {r2_train:<30.4f}{rmse_train:<12.4f}")
        print(f"Test:     {r2_test:<30.4f}{rmse_test:<12.4f}")
        print("\n\n")

    return r2_train, rmse_train, r2_test, rmse_test

In [ ]:
# Run the 'prophet_metric_train_test' function

prophet_metric_train_test(train, test_df, forecast_train, forecast_test)

In [ ]:
# Plot predicted & actual vs date

# Plot for training data
plt.figure(figsize=(10, 6))
plt.plot(train['ds'], train['y'], label='Actual', color='blue')
plt.plot(forecast_train['ds'], forecast_train['yhat'], label='Forecast', color='orange')
plt.fill_between(forecast_train['ds'], forecast_train['yhat_lower'], forecast_train['yhat_upper'], color='orange', alpha=0.2, label='Uncertainty Interval')
plt.xlabel('Date')
plt.ylabel('Value')
plt.title('Training Data - Actual vs Forecast')
plt.legend()
plt.show()

# Plot for test data
plt.figure(figsize=(10, 6))
plt.plot(test_df['ds'], test_df['y'], label='Actual', color='blue')
plt.plot(forecast_test['ds'], forecast_test['yhat'], label='Forecast', color='orange')
plt.fill_between(forecast_test['ds'], forecast_test['yhat_lower'], forecast_test['yhat_upper'], color='orange', alpha=0.2, label='Uncertainty Interval')
plt.xlabel('Date')
plt.ylabel('Value')
plt.title('Test Data - Actual vs Forecast')
plt.legend()
plt.show()

In [ ]:
# Plot predicted vs. actual

# Draw a scatter plot with regression line

# Plot for training data
plt.figure(figsize=(6, 6))
plt.scatter(train['y'], forecast_train['yhat'], alpha=0.5, s=20, color='navy')
plt.plot(np.linspace(np.min(train['y']), np.max(train['y']), 100), np.linspace(np.min(train['y']), np.max(train['y']), 100), color='red', linestyle='--')
plt.xlabel("Actual")
plt.ylabel("Forecast")
plt.title("Training Data: Actual vs Forecast")
plt.grid(True)
plt.show()

# Plot for test data
plt.figure(figsize=(6, 6))
plt.scatter(test_df['y'], forecast_test['yhat'], alpha=0.5, s=20, color='navy')
plt.plot(np.linspace(np.min(test_df['y']), np.max(test_df['y']), 100), np.linspace(np.min(test_df['y']), np.max(test_df['y']), 100), color='red', linestyle='--')
plt.xlabel("Actual")
plt.ylabel("Forecast")
plt.title("Test Data: Actual vs Forecast")
plt.grid(True)
plt.show()

### Function for the drawing of scatter-plot of pred. vs actual

In [ ]:
# Function for the plotting of predicted vs. actual

# Draw a scatter plot with regression line and R2 and RMSE values

def plot_prophet_pred_actu(train, test_df, forecast_train, forecast_test, title_train="Training Data: Actual vs Forecast", title_test="Test Data: Actual vs Forecast", anno_relative_x=0.03, anno_relative_y=0.97):

    # Prepare for annotations
    r2_train, rmse_train, r2_test, rmse_test = prophet_metric_train_test(train, test_df, forecast_train, forecast_test)

    # Relative coordinates (anno_relative_x, anno_relative_y) for the annotation of the plot
    relative_x = anno_relative_x
    relative_y = anno_relative_y

    offset = 0  # needed to calculate annotation text's offset from the xy data point 

    # Plot for training data
    plt.figure(figsize=(6, 6))
    plt.scatter(train['y'], forecast_train['yhat'], alpha=0.5, s=20, color='navy')
    plt.plot(np.linspace(np.min(train['y']), np.max(train['y']), 100), np.linspace(np.min(train['y']), np.max(train['y']), 100), color='red', linestyle='--')
    plt.xlabel("Actual")
    plt.ylabel("Forecast")
    plt.title(title_train)

    # Convert relative coordinates to data coordinates
    # data_x = plt.gca().get_xlim()[0] + relative_x * (plt.gca().get_xlim()[1] - plt.gca().get_xlim()[0])
    # data_y = plt.gca().get_ylim()[0] + relative_y * (plt.gca().get_ylim()[1] - plt.gca().get_ylim()[0])
    data_x = np.min(train['y']) + relative_x * (np.max(train['y']) - np.min(train['y']))
    data_y = np.min(train['y']) + relative_y * (np.max(train['y']) - np.min(train['y']))

    # # Get the bounding box of the main plotting area
    # bbox = plt.gca().get_window_extent()

    # # Calculate annotation coordinates based on the bounding box
    # data_x = bbox.x0 + relative_x * bbox.width
    # data_y = bbox.y0 + relative_y * bbox.height

    # # For debug
    # print(f"Bbox: bbox_width={bbox.width}, bbox_height={bbox.height}")
    print(f"Annotation's location: x={data_x}, y={data_y}")

    # Add a text annotation at the relative location
    plt.annotate(
        f"R2 score: {r2_train:6.4f}\nRMSE:      {rmse_train:6.4f}",  # Text to display
        xy=(data_x, data_y),   # Data point (x, y) to annotate (corresponding coordinate system is determined by xycoords)
        xytext =(0.5 * offset, -offset),        # Location of the text (coordinate system is determined by textcoords)
        # arrowprops=dict(facecolor='black', arrowstyle='->'),  # Arrow properties
        xycoords='data',       # Use data coordinates for xy
        textcoords='offset pixels',  # Offset (in pixels) from the xy value
        ha='left',           # Horizontal alignment of text
        va='top'            # Vertical alignment of text
    )

    plt.grid(True)
    plt.show()


    # Plot for test data
    plt.figure(figsize=(6, 6))
    plt.scatter(test_df['y'], forecast_test['yhat'], alpha=0.5, s=20, color='navy')
    plt.plot(np.linspace(np.min(test_df['y']), np.max(test_df['y']), 100), np.linspace(np.min(test_df['y']), np.max(test_df['y']), 100), color='red', linestyle='--')
    plt.xlabel("Actual")
    plt.ylabel("Forecast")
    plt.title(title_test)

    # Convert relative coordinates to data coordinates
    # data_x = plt.gca().get_xlim()[0] + relative_x * (plt.gca().get_xlim()[1] - plt.gca().get_xlim()[0])
    # data_y = plt.gca().get_ylim()[0] + relative_y * (plt.gca().get_ylim()[1] - plt.gca().get_ylim()[0])
    data_x = np.min(test_df['y']) + relative_x * (np.max(test_df['y']) - np.min(test_df['y']))
    data_y = np.min(test_df['y']) + relative_y * (np.max(test_df['y']) - np.min(test_df['y']))

    # # Get the bounding box of the main plotting area
    # bbox = plt.gca().get_window_extent()

    # # Calculate annotation coordinates based on the bounding box
    # data_x = bbox.x0 + relative_x * bbox.width
    # data_y = bbox.y0 + relative_y * bbox.height

    # # For debug
    # print(f"Bbox: {bbox}")
    print(f"Annotation's location: x={data_x}, y={data_y}")

    # Add a text annotation at the relative location
    plt.annotate(
        f"R2 score: {r2_test:6.4f}\nRMSE:      {rmse_test:6.4f}",  # Text to display
        xy=(data_x, data_y),   # Data point (x, y) to annotate (corresponding coordinate system is determined by xycoords)
        xytext =(0.5 * offset, -offset),        # Location of the text (coordinate system is determined by textcoords)
        # arrowprops=dict(facecolor='black', arrowstyle='->'),  # Arrow properties
        xycoords='data',       # Use data coordinates for xy
        textcoords='offset pixels',  # Offset (in pixels) from the xy value
        ha='left',           # Horizontal alignment of text
        va='top'            # Vertical alignment of text
    )
    
    plt.grid(True)
    plt.show()




In [ ]:
# Run the function 'plot_prophet_pred_actu'

plot_prophet_pred_actu(train, test_df, forecast_train, forecast_test)

### Play with 'n_timeunits_forecast'

In [ ]:
n_list = [6, 12, 24, 36, 48, 60]

for n in n_list:
    print("====="*20)
    print(f"*****  Making forecasts for {n} time units...  *****")

    forecast_af, future_af, train, test_df, forecast_train, forecast_test = prophet_af(df_m, seasonality_mode='multiplicative', yearly_seasonality=4, changepoint_prior_scale=0.5, list_added_regressor=['solar_gw', 'mean_air_temperature_de_c', 'mean_sunshine_duration_de_minute', 'mean_precipitation_de_mm'], freq='M', n_timeunits_forecast=n)

    plot_prophet_forecast(train, test_df, forecast_train, forecast_test, "Date", "Solar energy generation [TWh]", "Germany's monthly solar energy generation: actual vs. predicted")

    prophet_metric_train_test(train, test_df, forecast_train, forecast_test)

# Revisit 'df_d'

In [ ]:
df_d.head(2)

In [ ]:
df_d.info()

In [ ]:
n_list = [180, 365]

for n in n_list:
    print("====="*20)
    print(f"*****  Making forecasts for {n} time units...  *****")

    forecast_af, future_af, train, test_df, forecast_train, forecast_test = prophet_af(df_d, seasonality_mode='multiplicative', yearly_seasonality=4, changepoint_prior_scale=0.03, list_added_regressor=[], freq='D', n_timeunits_forecast=n)

    plot_prophet_forecast(train, test_df, forecast_train, forecast_test, "Date", "Solar energy generation [GWh]", "Germany's daily solar energy generation: actual vs. predicted")

    prophet_metric_train_test(train, test_df, forecast_train, forecast_test)

# 'df_d_endjune' (end date: 2023-06-30)

In [ ]:
print(df_d.shape)
df_d.tail(2)

In [ ]:
df_d.info()

In [ ]:
# Define the end date
end_date = pd.to_datetime('2023-06-30')

# Filter rows based on the 'ds' column
df_d_endjune = df_d[df_d['ds'] <= end_date]

df_d_endjune = df_d_endjune.reset_index(drop=True)

In [ ]:
print(df_d_endjune.shape)
df_d_endjune.tail(2)

# Import 'weather_daily_10loc.csv'

In [ ]:
df_d_w_orig = pd.read_csv('data/weather_daily_10loc.csv')

In [ ]:
df_d_w_orig.head(2)
df_d_w_orig.tail(5)
# df_d_w_orig.info()

In [ ]:
df_d_w_orig['dt_date'] = pd.to_datetime(df_d_w_orig['dt_date'])

# Define the end date
end_date = pd.to_datetime('2023-06-30')

# Filter rows based on the 'ds' column
df_d_w_endjune = df_d_w_orig[df_d_w_orig['dt_date'] <= end_date]

df_d_w_endjune = df_d_w_endjune.reset_index(drop=True)

In [ ]:
df_d_w_endjune.tail(2)
print(df_d_w_endjune.shape)

# 'df_dw_endjune'

## Drop duplicate columns

In [ ]:
columns_to_drop = ['dt_date.' + str(n) for n in range(1, 10)]
df_dw_endjune = df_d_w_endjune.drop(columns=columns_to_drop, inplace=False)
print(df_dw_endjune.shape)

In [ ]:
print(df_dw_endjune.columns)
df_dw_endjune.head(2)

## Calc aggregated values

### Calc mean
For: average temperature, norm temperature, max temperature, min temperature, sunshine duration, precipitation, and average wind speed

In [ ]:
# Stations from 'web_scraping_wetterzentrale.ipynb'
stations = [
    ("4625", "Schwerin", "sc"),
    ("1975", "Hamburg-Fuhlsbuettel", "hh"),
    ("7106", "Bielefeld-Deppendorf", "bi"),
    ("2667", "Koeln-Bonn", "kb"),
    ("1691", "Goettingen", "go"),
    ("1048", "Dresden-Klotzsche", "dr"),
    ("3668", "Nuernberg", "nu"),
    ("3244", "Memmingen", "me"),
    ("2712", "Konstanz", "ko"),
    ("1262", "Muenchen-Flughafen", "mu")    
]

list_station_id = []
list_station_city = []

for station_code, station_name, station_desc in stations:
    list_station_id.append(station_desc)
    list_station_city.append(station_name)

print(list_station_id)
print(list_station_city)

In [ ]:
# Define a list of str_start for all the 7 weather features
list_str_start = ['average_temp_c_', 'norm_temp_c_',
       'max_temp_c_', 'min_temp_c_', 'sunshine_hour_',
       'precipitation_mm_', 'average_wind_speed_mpers_']

In [ ]:
# Calculate means

# Find all columns that start with str_start
for str_start in list_str_start:
    wf_columns = [col for col in df_dw_endjune.columns if col.startswith(str_start)]

    # Calculate mean
    new_col_name = str_start + 'mean_de'
    df_dw_endjune[new_col_name] = df_dw_endjune[wf_columns].mean(axis=1)


In [ ]:
df_dw_endjune.tail(2)

In [ ]:
df_dw_endjune.describe().T

## Concatenate DataFrames

In [ ]:
df_d_all = pd.concat([df_d_endjune, df_dw_endjune], axis=1)

In [ ]:
df_d_all.head(2)

In [ ]:
info_describe(df_d_all, 2)

In [ ]:
print(df_d_all.shape)
print(df_d_all.columns)
list_columns_df_d_all = []
for item in df_d_all.columns:
    list_columns_df_d_all.append(item)

print(len(list_columns_df_d_all))
print(list_columns_df_d_all)


## Export 'df_d_all' to CSV

In [ ]:
# Export 'df_d_all' to CSV
# df_d_all.to_csv('./output/df_d_all.csv', sep=',', index=False)

# 'df_d_all': the final DataFrame for daily generation!

## Modeling without additional regressors!

In [ ]:
n_list = [180, 365, 730]

for n in n_list:
    print("====="*20)
    print(f"*****  Making forecasts for {n} time units...  *****")

    forecast_af, future_af, train, test_df, forecast_train, forecast_test = prophet_af(df_d_all, seasonality_mode='multiplicative', yearly_seasonality=4, changepoint_prior_scale=0.03, list_added_regressor=[], freq='D', n_timeunits_forecast=n)

    plot_prophet_forecast(train, test_df, forecast_train, forecast_test, "Date", "Solar energy generation [GWh]", "Germany's daily solar energy generation: actual vs. predicted")

    prophet_metric_train_test(train, test_df, forecast_train, forecast_test)

    plot_prophet_pred_actu(train, test_df, forecast_train, forecast_test)

## Modeling with 7 additional regressors!

In [ ]:
n_list = [180, 365, 730, 1461]

for n in n_list:
    print("====="*20)
    print(f"*****  Making forecasts for {n} time units...  *****")

    forecast_af, future_af, train, test_df, forecast_train, forecast_test = prophet_af(df_d_all, seasonality_mode='multiplicative', yearly_seasonality=4, changepoint_prior_scale=0.03, list_added_regressor=['average_temp_c_mean_de', 'norm_temp_c_mean_de', 'max_temp_c_mean_de', 'min_temp_c_mean_de', 'sunshine_hour_mean_de', 'precipitation_mm_mean_de', 'average_wind_speed_mpers_mean_de'], freq='D', n_timeunits_forecast=n)

    plot_prophet_forecast(train, test_df, forecast_train, forecast_test, "Date", "Solar energy generation [GWh]", "Germany's daily solar energy generation: actual vs. predicted")

    prophet_metric_train_test(train, test_df, forecast_train, forecast_test)

    plot_prophet_pred_actu(train, test_df, forecast_train, forecast_test)

## "Feature importance" assessment 

### All possible combinations of additional regressors

In [ ]:
list_weather_regressors=['average_temp_c_mean_de', 'norm_temp_c_mean_de', 'max_temp_c_mean_de', 'min_temp_c_mean_de', 'sunshine_hour_mean_de', 'precipitation_mm_mean_de', 'average_wind_speed_mpers_mean_de']

In [ ]:
# Generate all possible subsets of the original list
subsets = []
for r in range(1, len(list_weather_regressors) + 1):
    subsets.extend(combinations(list_weather_regressors, r))

# Filter out subsets with duplicate elements
filtered_subsets = [subset for subset in subsets if len(set(subset)) == len(subset)]

# Print the filtered subsets
for subset in filtered_subsets:
    print(subset)

In [ ]:
# Group the constructed lists based on the number of different elements

# Generate all possible subsets of the original list
subsets = []
for r in range(1, len(list_weather_regressors) + 1):
    subsets.extend(combinations(list_weather_regressors, r))

# Filter out subsets with duplicate elements
filtered_subsets = [subset for subset in subsets if len(set(subset)) == len(subset)]

# Group subsets by the number of elements
grouped_subsets = defaultdict(list)
for subset in filtered_subsets:
    num_elements = len(subset)
    grouped_subsets[num_elements].append(subset)

# Print the grouped subsets
for num_elements, subsets in grouped_subsets.items():
    print(f"Number of Elements: {num_elements}")
    for subset in subsets:
        print(subset)
    print()


In [ ]:
# Construct lists of lists for subsets grouped by the number of elements 

# Generate all possible subsets of the original list
subsets = []
for r in range(1, len(list_weather_regressors) + 1):
    subsets.extend(combinations(list_weather_regressors, r))

# Filter out subsets with duplicate elements
filtered_subsets = [subset for subset in subsets if len(set(subset)) == len(subset)]

# Group subsets by the number of elements
grouped_subsets = {}
for subset in filtered_subsets:
    num_elements = len(subset)
    if num_elements not in grouped_subsets:
        grouped_subsets[num_elements] = []
    grouped_subsets[num_elements].append(list(subset))

# Convert the dictionary values to lists of lists
grouped_lists = list(grouped_subsets.values())

# Print the grouped lists
dict_num_weather_elements = {}
for i, group in enumerate(grouped_lists, start=1):
    print(f"Group {i} ({len(group[0])} elements):")
    print(group)
    # print(grouped_lists[i-1])
    dict_num_weather_elements[i] = group
    print()

# pprint.pprint(dict_num_weather_elements)
# pprint.pprint(dict_num_weather_elements[1])


### 0 additional regressor

In [ ]:
n_list = [180, 365, 730, 1461]

list_weather_feature_combo = [[]]    # List of lists

list_num_weather_features_in_use = [] # To be stored in a result DataFrame
list_num_timeunits = [] # To be stored in a result DataFrame
list_used_weather_features = [] # To be stored in a result DataFrame
list_r2_score_train = []  # To be stored in a result DataFrame
list_r2_score_test = []  # To be stored in a result DataFrame
list_rmse_train = []  # To be stored in a result DataFrame
list_rmse_test = []  # To be stored in a result DataFrame

for n in n_list:
    print("====="*20)

    for weather_feature_combo in list_weather_feature_combo:

        print("-----"*20)
        print(f"** Making forecasts for {n} time units...")
        print(f"** Weather features: {weather_feature_combo}...")
        print("-----"*12)

        list_num_weather_features_in_use.append(len(weather_feature_combo))
        list_num_timeunits.append(n)
        list_used_weather_features.append(weather_feature_combo)

        forecast_af, future_af, train, test_df, forecast_train, forecast_test = prophet_af(df_d_all, seasonality_mode='multiplicative', yearly_seasonality=4, changepoint_prior_scale=0.03, list_added_regressor=weather_feature_combo, freq='D', n_timeunits_forecast=n)

        plot_prophet_forecast(train, test_df, forecast_train, forecast_test, "Date", "Solar energy generation [GWh]", "Germany's daily solar energy generation: actual vs. predicted")

        r2_train, rmse_train, r2_test, rmse_test = prophet_metric_train_test(train, test_df, forecast_train, forecast_test)

        list_r2_score_train.append(r2_train) 
        list_r2_score_test.append(r2_test)
        list_rmse_train.append(rmse_train)
        list_rmse_test.append(rmse_test) 

        plot_prophet_pred_actu(train, test_df, forecast_train, forecast_test)

# Create a DataFrame to store added weather features and performance scores
df_0add_metric = pd.DataFrame({
    "number_of_weather_features": list_num_weather_features_in_use,
    "number_of_timeunits": list_num_timeunits,
    "weather_features_in_use": list_used_weather_features,
    "r2_score_training_data": list_r2_score_train,
    "r2_score_test_data": list_r2_score_test,
    "rmse_training_data": list_rmse_train,
    "rmse_test_data": list_rmse_test
})

df_0add_metric.head(100)

### 1 additional regressor

In [ ]:
n_list = [180, 365, 730, 1461]

list_weather_feature_combo = dict_num_weather_elements[1]    # List of lists

list_num_weather_features_in_use = [] # To be stored in a result DataFrame
list_num_timeunits = [] # To be stored in a result DataFrame
list_used_weather_features = [] # To be stored in a result DataFrame
list_r2_score_train = []  # To be stored in a result DataFrame
list_r2_score_test = []  # To be stored in a result DataFrame
list_rmse_train = []  # To be stored in a result DataFrame
list_rmse_test = []  # To be stored in a result DataFrame

for n in n_list:
    print("====="*20)

    for weather_feature_combo in list_weather_feature_combo:

        print("-----"*20)
        print(f"** Making forecasts for {n} time units...")
        print(f"** Weather features: {weather_feature_combo}...")
        print("-----"*12)

        list_num_weather_features_in_use.append(len(weather_feature_combo))
        list_num_timeunits.append(n)
        list_used_weather_features.append(weather_feature_combo)

        forecast_af, future_af, train, test_df, forecast_train, forecast_test = prophet_af(df_d_all, seasonality_mode='multiplicative', yearly_seasonality=4, changepoint_prior_scale=0.03, list_added_regressor=weather_feature_combo, freq='D', n_timeunits_forecast=n)

        plot_prophet_forecast(train, test_df, forecast_train, forecast_test, "Date", "Solar energy generation [GWh]", "Germany's daily solar energy generation: actual vs. predicted")

        r2_train, rmse_train, r2_test, rmse_test = prophet_metric_train_test(train, test_df, forecast_train, forecast_test)

        list_r2_score_train.append(r2_train) 
        list_r2_score_test.append(r2_test)
        list_rmse_train.append(rmse_train)
        list_rmse_test.append(rmse_test) 

        plot_prophet_pred_actu(train, test_df, forecast_train, forecast_test)

# Create a DataFrame to store added weather features and performance scores
df_1add_metric = pd.DataFrame({
    "number_of_weather_features": list_num_weather_features_in_use,
    "number_of_timeunits": list_num_timeunits,
    "weather_features_in_use": list_used_weather_features,
    "r2_score_training_data": list_r2_score_train,
    "r2_score_test_data": list_r2_score_test,
    "rmse_training_data": list_rmse_train,
    "rmse_test_data": list_rmse_test
})

df_1add_metric.head(100)

### 2 additional regressors

In [ ]:
# Construct a list of lists for subsets with 2 elements ('sunshine_hour_mean_de' + another weather feature)

list_weather_regressors=['average_temp_c_mean_de', 'norm_temp_c_mean_de', 'max_temp_c_mean_de', 'min_temp_c_mean_de', 'sunshine_hour_mean_de', 'precipitation_mm_mean_de', 'average_wind_speed_mpers_mean_de']

fixed_feature = 'sunshine_hour_mean_de'

# Remove fixed_feature from the list
list_without_fixed_feature = [item for item in list_weather_regressors if item != fixed_feature]

# Generate all possible pairs with fixed_feature and other elements
pairs_with_fixed_feature = [(fixed_feature, item) for item in list_without_fixed_feature]

# Create lists with fixed_feature as the first element
result_lists_fixed_plus_1_more_feature = [[first] + [rest] for first, rest in pairs_with_fixed_feature]

# Print the resulting lists
for result_list in result_lists_fixed_plus_1_more_feature:
    print(result_list)

print(result_lists_fixed_plus_1_more_feature)


In [ ]:
n_list = [180, 365, 730, 1461]

list_weather_feature_combo = result_lists_fixed_plus_1_more_feature    # List of lists

list_num_weather_features_in_use = [] # To be stored in a result DataFrame
list_num_timeunits = [] # To be stored in a result DataFrame
list_used_weather_features = [] # To be stored in a result DataFrame
list_r2_score_train = []  # To be stored in a result DataFrame
list_r2_score_test = []  # To be stored in a result DataFrame
list_rmse_train = []  # To be stored in a result DataFrame
list_rmse_test = []  # To be stored in a result DataFrame

for n in n_list:
    print("====="*20)

    for weather_feature_combo in list_weather_feature_combo:

        print("-----"*20)
        print(f"** Making forecasts for {n} time units...")
        print(f"** Weather features: {weather_feature_combo}...")
        print("-----"*12)

        list_num_weather_features_in_use.append(len(weather_feature_combo))
        list_num_timeunits.append(n)
        list_used_weather_features.append(weather_feature_combo)

        forecast_af, future_af, train, test_df, forecast_train, forecast_test = prophet_af(df_d_all, seasonality_mode='multiplicative', yearly_seasonality=4, changepoint_prior_scale=0.03, list_added_regressor=weather_feature_combo, freq='D', n_timeunits_forecast=n)

        plot_prophet_forecast(train, test_df, forecast_train, forecast_test, "Date", "Solar energy generation [GWh]", "Germany's daily solar energy generation: actual vs. predicted")

        r2_train, rmse_train, r2_test, rmse_test = prophet_metric_train_test(train, test_df, forecast_train, forecast_test)

        list_r2_score_train.append(r2_train) 
        list_r2_score_test.append(r2_test)
        list_rmse_train.append(rmse_train)
        list_rmse_test.append(rmse_test) 

        plot_prophet_pred_actu(train, test_df, forecast_train, forecast_test)

# Create a DataFrame to store added weather features and performance scores
df_2add_metric = pd.DataFrame({
    "number_of_weather_features": list_num_weather_features_in_use,
    "number_of_timeunits": list_num_timeunits,
    "weather_features_in_use": list_used_weather_features,
    "r2_score_training_data": list_r2_score_train,
    "r2_score_test_data": list_r2_score_test,
    "rmse_training_data": list_rmse_train,
    "rmse_test_data": list_rmse_test
})

df_2add_metric.head(100)

### 7 additional regressors

In [ ]:
n_list = [180, 365, 730, 1461]

list_weather_feature_combo = dict_num_weather_elements[7]    # List of lists

list_num_weather_features_in_use = [] # To be stored in a result DataFrame
list_num_timeunits = [] # To be stored in a result DataFrame
list_used_weather_features = [] # To be stored in a result DataFrame
list_r2_score_train = []  # To be stored in a result DataFrame
list_r2_score_test = []  # To be stored in a result DataFrame
list_rmse_train = []  # To be stored in a result DataFrame
list_rmse_test = []  # To be stored in a result DataFrame

for n in n_list:
    print("====="*20)

    for weather_feature_combo in list_weather_feature_combo:

        print("-----"*20)
        print(f"** Making forecasts for {n} time units...")
        print(f"** Weather features: {weather_feature_combo}...")
        print("-----"*12)

        list_num_weather_features_in_use.append(len(weather_feature_combo))
        list_num_timeunits.append(n)
        list_used_weather_features.append(weather_feature_combo)

        forecast_af, future_af, train, test_df, forecast_train, forecast_test = prophet_af(df_d_all, seasonality_mode='multiplicative', yearly_seasonality=4, changepoint_prior_scale=0.03, list_added_regressor=weather_feature_combo, freq='D', n_timeunits_forecast=n)

        plot_prophet_forecast(train, test_df, forecast_train, forecast_test, "Date", "Solar energy generation [GWh]", "Germany's daily solar energy generation: actual vs. predicted")

        plot_prophet_forecast_zoomin(train, test_df, forecast_train, forecast_test, "Date", "Solar energy generation [TWh]", "Germany's monthly solar energy generation: actual vs. predicted", 20, 'D')

        r2_train, rmse_train, r2_test, rmse_test = prophet_metric_train_test(train, test_df, forecast_train, forecast_test)

        list_r2_score_train.append(r2_train) 
        list_r2_score_test.append(r2_test)
        list_rmse_train.append(rmse_train)
        list_rmse_test.append(rmse_test) 

        plot_prophet_pred_actu(train, test_df, forecast_train, forecast_test)

# Create a DataFrame to store added weather features and performance scores
df_7add_metric = pd.DataFrame({
    "number_of_weather_features": list_num_weather_features_in_use,
    "number_of_timeunits": list_num_timeunits,
    "weather_features_in_use": list_used_weather_features,
    "r2_score_training_data": list_r2_score_train,
    "r2_score_test_data": list_r2_score_test,
    "rmse_training_data": list_rmse_train,
    "rmse_test_data": list_rmse_test
})

df_7add_metric.head(100)

### Combine 4 metric DataFrames

In [ ]:
# Concatenate DataFrames along the y direction
df_0127_metric = pd.concat([df_0add_metric, df_1add_metric, df_2add_metric, df_7add_metric], axis=0)

df_0127_metric.reset_index(drop=True, inplace=True)

df_0127_metric.head(100)

In [ ]:
print(df_0127_metric.columns)

In [ ]:
# Sort the DataFrame first by 'rmse_test_data' in descending order, then by 'r2_score_test_data' in descending order
sorted_df_metric = df_0127_metric.sort_values(by=['rmse_test_data', 'r2_score_test_data'], ascending=[True, False])
sorted_df_metric.head(100)